# ResNet: Deep Residual Learning for Image Recognition — Implementation / 구현

**Paper**: He, K., Zhang, X., Ren, S., & Sun, J. (2015). *Deep Residual Learning for Image Recognition*. arXiv:1512.03385 (CVPR 2016).

This notebook implements the core ideas of ResNet from scratch in PyTorch, and empirically demonstrates the **degradation problem** and how residual connections resolve it.

이 노트북은 PyTorch로 ResNet의 핵심 아이디어를 바닥부터 구현하고, **degradation 문제**와 이를 residual 연결이 어떻게 해결하는지 실험으로 보입니다.

## Contents / 목차
1. **Residual Block** — Basic (2-layer) and Bottleneck (3-layer) 구현
2. **Building a Full ResNet** — ResNet-18/34/50 for CIFAR-like inputs
3. **Plain vs Residual: Degradation Demo** — 동일 데이터에서 plain vs resnet 훈련 오차 비교
4. **Gradient Flow Analysis** — Skip connection이 기울기 분포에 미치는 영향 시각화
5. **Residual Response Analysis** — 학습된 잔차 함수의 크기 분석 (Figure 7 재현)
6. **Summary Table** — 원 논문 vs 현대 실용 비교

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## Part 1: Residual Block / 잔차 블록

Paper Eq. (1): $\mathbf{y} = F(\mathbf{x}, \{W_i\}) + \mathbf{x}$

When dimensions change, use projection shortcut (Eq. 2): $\mathbf{y} = F(\mathbf{x}) + W_s \mathbf{x}$

원 논문의 Figure 2(basic) / Figure 5(bottleneck) 구조를 그대로 구현합니다.

We implement both designs: (a) **Basic block** — two 3×3 convs; (b) **Bottleneck block** — 1×1 → 3×3 → 1×1 channel-reducing design used in ResNet-50/101/152.

In [ ]:
class BasicBlock(nn.Module):
    """2-layer residual block used in ResNet-18/34.

    Computes y = ReLU(F(x) + shortcut(x)) where F is two 3x3 convs
    each followed by BatchNorm (post-activation / v1 style).
    """
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        # Projection shortcut when dim changes; identity otherwise.
        if stride != 1 or in_ch != out_ch * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch * self.expansion, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_ch * self.expansion)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)  # Paper Eq. (1)
        return F.relu(out, inplace=True)


class BottleneckBlock(nn.Module):
    """3-layer bottleneck block used in ResNet-50/101/152.

    1x1 (reduce) -> 3x3 (main) -> 1x1 (restore to 4x channels).
    This is the parameter-efficient design that makes very deep
    networks tractable (see notes Section 4.3).
    """
    expansion = 4

    def __init__(self, in_ch, mid_ch, stride=1):
        super().__init__()
        out_ch = mid_ch * self.expansion
        self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.conv2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_ch)
        self.conv3 = nn.Conv2d(mid_ch, out_ch, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)

        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.bn3(self.conv3(out))
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


# Quick sanity check: 64-channel input through a basic block keeps shape.
demo_in = torch.randn(2, 64, 32, 32)
print('BasicBlock(64->64, stride=1)  out:', BasicBlock(64, 64).eval()(demo_in).shape)
print('BasicBlock(64->128, stride=2) out:', BasicBlock(64, 128, stride=2).eval()(demo_in).shape)
print('Bottleneck(256->64*4)         out:', BottleneckBlock(256, 64).eval()(torch.randn(2, 256, 16, 16)).shape)

### FLOPs comparison: Basic vs Bottleneck / 계산량 비교

논문 §3.3에서 bottleneck의 이점은 "같은 계산 예산으로 더 깊은 네트워크"입니다. 256채널 입력 기준으로 직접 계산해봅니다.

For 256-channel input, we reproduce the FLOP accounting from the paper.

In [ ]:
def conv_flops_per_pixel(in_c, out_c, k):
    """FLOPs per output pixel for a conv (ignoring bias)."""
    return k * k * in_c * out_c

basic_256 = 2 * conv_flops_per_pixel(256, 256, 3)
bottle_256 = (conv_flops_per_pixel(256, 64, 1)
              + conv_flops_per_pixel(64, 64, 3)
              + conv_flops_per_pixel(64, 256, 1))

print(f'Basic (256->256, two 3x3): {basic_256:>10,} FLOPs/pixel')
print(f'Bottleneck (256->64->256): {bottle_256:>10,} FLOPs/pixel')
print(f'Bottleneck is ~{basic_256 / bottle_256:.1f}x cheaper — exactly why ResNet-50+ feasible.')

## Part 2: Full ResNet / 전체 ResNet 조립

논문 Table 1의 구조를 구현합니다. CIFAR 크기(32×32) 실험을 위해 conv1을 3×3으로 단순화하고 maxpool을 제거한 버전도 함께 제공합니다 (논문 §4.2).

We build ResNet-18/34/50 for ImageNet-sized input, and also the CIFAR variant (3×3 initial conv, no max-pool, 3 stages of $n$ blocks → depth = $6n+2$).

In [ ]:
class ResNet(nn.Module):
    """Generic ResNet for ImageNet-style input (224x224).

    Args:
        block: BasicBlock or BottleneckBlock.
        layers: number of blocks per stage, e.g. [2,2,2,2] for ResNet-18.
        num_classes: output classes.
    """
    def __init__(self, block, layers, num_classes=1000):
        super().__init__()
        self.in_ch = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1 = self._make_stage(block,  64, layers[0], stride=1)
        self.layer2 = self._make_stage(block, 128, layers[1], stride=2)
        self.layer3 = self._make_stage(block, 256, layers[2], stride=2)
        self.layer4 = self._make_stage(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        # He initialization (paper §3.4).
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_stage(self, block, mid_ch, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        blocks = []
        for s in strides:
            blocks.append(block(self.in_ch, mid_ch, stride=s))
            self.in_ch = mid_ch * block.expansion
        return nn.Sequential(*blocks)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).flatten(1)
        return self.fc(x)


def resnet18(num_classes=1000):
    return ResNet(BasicBlock,      [2, 2, 2, 2], num_classes)

def resnet34(num_classes=1000):
    return ResNet(BasicBlock,      [3, 4, 6, 3], num_classes)

def resnet50(num_classes=1000):
    return ResNet(BottleneckBlock, [3, 4, 6, 3], num_classes)


for name, factory in [('ResNet-18', resnet18), ('ResNet-34', resnet34), ('ResNet-50', resnet50)]:
    m = factory()
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name:<10} params: {n_params/1e6:6.2f} M')

## Part 3: Plain vs Residual — Degradation Demo / 퇴화 현상 시연

논문의 핵심 주장: **plain 네트워크는 깊이가 증가할수록 훈련 오차가 증가하지만, residual 네트워크는 그렇지 않다.** CIFAR-10 전체를 훈련하기에는 시간이 오래 걸리므로, 이 섹션에서는 **합성 데이터**를 사용해 degradation 현상을 짧은 시간 내에 재현합니다.

We reproduce the paper's key claim on synthetic data for speed: plain deep networks show worse *training* loss than shallow ones, while residual nets improve monotonically with depth.

In [ ]:
class TinyResBlock(nn.Module):
    """Small 1D residual block for synthetic experiment."""
    def __init__(self, dim, use_skip=True):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.use_skip = use_skip

    def forward(self, x):
        out = F.relu(self.bn1(self.fc1(x)), inplace=True)
        out = self.bn2(self.fc2(out))
        if self.use_skip:
            out = out + x  # residual
        return F.relu(out, inplace=True)


class TinyNet(nn.Module):
    """Deep stack of TinyResBlocks, with or without skip connections."""
    def __init__(self, dim, depth, use_skip):
        super().__init__()
        self.blocks = nn.Sequential(*[TinyResBlock(dim, use_skip) for _ in range(depth)])
        self.head = nn.Linear(dim, 1)

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(-1)


def make_synthetic(n=4096, dim=32):
    """Simple regression target: a mildly nonlinear function of x."""
    x = torch.randn(n, dim)
    w = torch.randn(dim, dim) * 0.3
    y = torch.tanh(x @ w).sum(dim=1) + 0.1 * torch.randn(n)
    return x, y


def train_model(net, x, y, epochs=40, lr=0.05):
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    losses = []
    bs = 256
    for ep in range(epochs):
        perm = torch.randperm(len(x))
        ep_loss = 0.0
        n_batches = 0
        for i in range(0, len(x), bs):
            idx = perm[i:i+bs]
            xb, yb = x[idx], y[idx]
            pred = net(xb)
            loss = F.mse_loss(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            ep_loss += loss.item()
            n_batches += 1
        losses.append(ep_loss / n_batches)
    return losses


DIM = 32
DEPTHS = [4, 10, 20, 40]
x_train, y_train = make_synthetic(n=4096, dim=DIM)

plain_curves, resnet_curves = {}, {}
for d in DEPTHS:
    torch.manual_seed(0)
    plain = TinyNet(DIM, d, use_skip=False)
    plain_curves[d] = train_model(plain, x_train, y_train)

    torch.manual_seed(0)
    resn = TinyNet(DIM, d, use_skip=True)
    resnet_curves[d] = train_model(resn, x_train, y_train)
    print(f'depth={d:>3}  plain final loss={plain_curves[d][-1]:.4f}   resnet final loss={resnet_curves[d][-1]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for d in DEPTHS:
    axes[0].plot(plain_curves[d],  label=f'{d} layers')
    axes[1].plot(resnet_curves[d], label=f'{d} layers')

axes[0].set_title('Plain network — training loss\n(deeper is WORSE — degradation)')
axes[1].set_title('Residual network — training loss\n(deeper is BETTER or equal)')
for ax in axes:
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend(); ax.grid(alpha=0.3)
    ax.set_yscale('log')
plt.tight_layout(); plt.show()

**관찰 / Observation**: plain network에서 40층이 4층보다 훈련 손실이 더 나쁘거나 비슷한 수준에 머뭅니다 (이 논문 Figure 1과 같은 현상). ResNet에서는 깊이가 증가해도 최소한 얕은 네트워크 성능은 유지되거나 더 좋아집니다. 이것이 "identity가 $F=0$으로 공짜"라는 주장의 경험적 증거입니다.

In the plain network, 40 layers trains **worse or no better** than 4 — exactly the degradation phenomenon of Figure 1. In ResNet, deeper is at least as good — empirical validation of "identity is trivially $F=0$."

## Part 4: Gradient Flow Analysis / 기울기 흐름 분석

Notes §4.4: 잔차 연결은 역전파 시 "+1" 항을 보장합니다. 초기화된 깊은 네트워크에 무작위 loss를 역전파했을 때, **각 층이 받는 기울기 norm**을 plain vs ResNet으로 비교합니다.

We compare per-layer gradient norms in a freshly initialized deep plain vs residual network. Residual skip connections preserve gradient magnitude across layers; plain networks attenuate (or amplify) it unpredictably.

In [ ]:
def layerwise_grad_norms(net, x, y):
    """Return list of gradient L2-norms for each block's fc1 weight."""
    net.zero_grad()
    loss = F.mse_loss(net(x), y)
    loss.backward()
    norms = []
    for blk in net.blocks:
        norms.append(blk.fc1.weight.grad.norm().item())
    return norms


DEPTH = 40
torch.manual_seed(1)
plain_deep = TinyNet(DIM, DEPTH, use_skip=False).train()
torch.manual_seed(1)
res_deep = TinyNet(DIM, DEPTH, use_skip=True).train()

xb = torch.randn(256, DIM)
yb = torch.randn(256)

plain_grads = layerwise_grad_norms(plain_deep, xb, yb)
res_grads = layerwise_grad_norms(res_deep, xb, yb)

plt.figure(figsize=(10, 5))
plt.plot(plain_grads, 'o-', label='Plain')
plt.plot(res_grads,   's-', label='ResNet (skip)')
plt.yscale('log')
plt.xlabel('Block index (0 = closest to input)')
plt.ylabel('Gradient L2 norm (log)')
plt.title(f'Per-block gradient norm in a {DEPTH}-layer net (at initialization)')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f'Plain  — early-layer grad / late-layer grad ratio = {plain_grads[0] / plain_grads[-1]:.2e}')
print(f'ResNet — early-layer grad / late-layer grad ratio = {res_grads[0] / res_grads[-1]:.2e}')

**관찰 / Observation**: Plain 네트워크는 초기 층으로 갈수록 기울기가 크게 감쇠(혹은 폭주)합니다. ResNet은 skip 덕분에 기울기 크기가 네트워크 전체에 걸쳐 **훨씬 균일**하게 유지됩니다 — 이것이 1000층급 네트워크 학습을 가능하게 하는 핵심 성질입니다.

Plain-net gradients attenuate (or amplify) dramatically across depth, while ResNet keeps them near-constant — the property that enables training 1000-layer networks.

## Part 5: Residual Response Analysis / 잔차 응답 분석 (Figure 7 재현)

논문 Figure 7: 훈련된 ResNet의 각 블록에서 **잔차 $F(x)$의 표준편차**가 plain 네트워크의 대응 응답보다 작다. 이는 "대부분의 블록이 입력에 작은 보정만 가한다"는 의미로, iterative refinement 관점의 근거입니다.

Reproduce Figure 7: after training, $F(x)$'s per-block std in ResNet is smaller than the corresponding activation std in plain networks — each block adds only a small correction.

In [ ]:
# Train both nets briefly and measure per-block residual response magnitude.
torch.manual_seed(2)
plain_net = TinyNet(DIM, 20, use_skip=False)
torch.manual_seed(2)
res_net   = TinyNet(DIM, 20, use_skip=True)

_ = train_model(plain_net, x_train, y_train, epochs=20)
_ = train_model(res_net,   x_train, y_train, epochs=20)


def block_response_stds(net, x):
    """For each block, compute std of F(x) (the non-skip branch output)."""
    net.eval()
    stds = []
    h = x
    with torch.no_grad():
        for blk in net.blocks:
            # F(x): the conv/BN/ReLU branch BEFORE the skip add.
            pre = blk.bn2(blk.fc2(F.relu(blk.bn1(blk.fc1(h)), inplace=False)))
            stds.append(pre.std().item())
            h = blk(h)
    return stds


plain_resp = block_response_stds(plain_net, x_train[:512])
res_resp   = block_response_stds(res_net,   x_train[:512])

plt.figure(figsize=(10, 5))
plt.plot(plain_resp, 'o-', label='Plain: std of block output F(x)')
plt.plot(res_resp,   's-', label='ResNet: std of residual F(x)')
plt.xlabel('Block index')
plt.ylabel('Response std')
plt.title('Per-block response magnitude (paper Figure 7 analogue)')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f'Mean response std — Plain : {np.mean(plain_resp):.3f}')
print(f'Mean response std — ResNet: {np.mean(res_resp):.3f}')

**관찰 / Observation**: 훈련된 ResNet의 **잔차 $F(x)$의 std는 plain 네트워크의 해당 층 출력 std보다 작은 경향**이 있습니다 (Figure 7의 아이디어). 이는 "각 블록은 입력을 통째로 재생성하지 않고, 작은 보정만 추가한다"는 현대적 해석(iterative refinement — Greff et al. 2017)의 경험적 근거입니다.

Trained residual branches $F(x)$ tend to have **smaller std** than plain-net activations — residual blocks apply small corrections rather than rewriting their input. This underpins the "iterative refinement" interpretation (Greff et al. 2017).

## Part 6: Post-activation (v1) vs Pre-activation (v2) / 사후 vs 사전 활성화

briefing Q2에서 논의한 바와 같이, 1년 뒤 He et al. (2016) ResNet v2는 **덧셈 후 ReLU를 제거**하고 activation을 블록 내부로 이동시켰습니다. 두 변형을 직접 구현하여 기울기 흐름 차이를 비교합니다.

As discussed in briefing Q2, ResNet v2 (He 2016) removes the ReLU after the addition. Here we implement both variants and compare gradient purity.

In [ ]:
class BasicBlockV1(nn.Module):
    """Post-activation: Conv-BN-ReLU-Conv-BN + skip → ReLU (this paper, 2015)."""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim); self.bn2 = nn.BatchNorm1d(dim)

    def forward(self, x):
        out = F.relu(self.bn1(self.fc1(x)), inplace=True)
        out = self.bn2(self.fc2(out))
        return F.relu(out + x, inplace=True)


class BasicBlockV2(nn.Module):
    """Pre-activation: BN-ReLU-Conv-BN-ReLU-Conv + skip (He 2016).
    Identity path is purely linear."""
    def __init__(self, dim):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(dim); self.fc1 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim); self.fc2 = nn.Linear(dim, dim)

    def forward(self, x):
        out = self.fc1(F.relu(self.bn1(x), inplace=False))
        out = self.fc2(F.relu(self.bn2(out), inplace=False))
        return out + x  # NO activation after add


class TinyNetV(nn.Module):
    def __init__(self, dim, depth, block_cls):
        super().__init__()
        self.blocks = nn.Sequential(*[block_cls(dim) for _ in range(depth)])
        self.head = nn.Linear(dim, 1)
    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(-1)


torch.manual_seed(3)
net_v1 = TinyNetV(DIM, 60, BasicBlockV1).train()
torch.manual_seed(3)
net_v2 = TinyNetV(DIM, 60, BasicBlockV2).train()


def grad_per_block(net, x, y, weight_attr='fc1'):
    net.zero_grad()
    F.mse_loss(net(x), y).backward()
    return [getattr(b, weight_attr).weight.grad.norm().item() for b in net.blocks]

xb2 = torch.randn(256, DIM); yb2 = torch.randn(256)
g_v1 = grad_per_block(net_v1, xb2, yb2)
g_v2 = grad_per_block(net_v2, xb2, yb2)

plt.figure(figsize=(10, 5))
plt.plot(g_v1, 'o-', label='Post-activation (v1)')
plt.plot(g_v2, 's-', label='Pre-activation (v2, no ReLU after add)')
plt.yscale('log')
plt.xlabel('Block index'); plt.ylabel('Gradient L2 norm (log)')
plt.title('v1 vs v2 — gradient uniformity across 60 blocks')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f'v1 ratio (first / last) = {g_v1[0] / g_v1[-1]:.2e}')
print(f'v2 ratio (first / last) = {g_v2[0] / g_v2[-1]:.2e}')

**관찰 / Observation**: v2는 v1보다 기울기 분포가 **더 균일**합니다. 덧셈 후에 ReLU가 없어 identity 경로가 순수하게 선형이기 때문입니다. 이 차이가 누적되어 1000층급 네트워크에서 v2가 v1보다 훨씬 잘 학습되는 이유입니다 (He 2016).

V2 shows more uniform gradients than v1 — its identity path is purely linear (no ReLU after the add). This is why v2 scales to 1000-layer networks (He 2016).

## Summary / 요약

| Concept / 개념 | This Paper (2015) / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Skip connection | Identity shortcut: $y = F(x) + x$ | Ubiquitous — in every ResNet variant, Transformer, diffusion model |
| Degradation fix | Residual reformulation $F(x) = H(x) - x$ | Same idea; now considered basic architectural hygiene |
| Dimension matching | (A) zero-padding (B) 1×1 projection (C) all-projection | Always (B) projection-only in practice (torchvision default) |
| Deep bottleneck | 1×1 → 3×3 → 1×1 block, 4× channel expansion | Still standard in ResNet-50/101/152 and derivatives (ResNeXt) |
| Activation placement | Post-activation (ReLU after add) | Pre-activation (ResNet v2, 2016) preferred for very deep nets; pre-norm is the Transformer analogue |
| Regularization | BN only, no dropout | + Stochastic Depth, Label Smoothing, MixUp, RandAugment |
| Normalization | BatchNorm | BN for CNNs; LayerNorm/RMSNorm for Transformers |
| Ensembling | 6-model ensemble for 3.57% | Single model + self-distillation / EMA reaches similar gains |

### Key insights reproduced in this notebook / 이 노트북에서 재현된 핵심 인사이트
- **Part 3**: Plain 깊은 네트워크의 degradation 현상 직접 관찰.
- **Part 4**: Skip connection이 기울기를 층별로 균일하게 유지함을 시각화.
- **Part 5**: 훈련된 잔차 $F(x)$가 실제로 작은 응답을 가진다는 Figure 7 아이디어 확인.
- **Part 6**: v1 vs v2 (post- vs pre-activation) 기울기 균일성 차이 시연.

이 네 가지가 "왜 ResNet이 깊이의 장벽을 넘었는가"를 정량적으로 설명하는 증거입니다.

Together these four experiments quantitatively explain why ResNet broke the depth barrier that stood for three years after AlexNet.